In [0]:
%sql
CREATE CATALOG IF NOT EXISTS flightops_dev
  MANAGED LOCATION 'abfss://gold@stflightopsdevuv0zba.dfs.core.windows.net/_uc_catalog';

CREATE SCHEMA IF NOT EXISTS flightops_dev.bronze
  MANAGED LOCATION 'abfss://bronze@stflightopsdevuv0zba.dfs.core.windows.net/_tables';
CREATE SCHEMA IF NOT EXISTS flightops_dev.silver
  MANAGED LOCATION 'abfss://silver@stflightopsdevuv0zba.dfs.core.windows.net/_tables';
CREATE SCHEMA IF NOT EXISTS flightops_dev.gold
  MANAGED LOCATION 'abfss://gold@stflightopsdevuv0zba.dfs.core.windows.net/_tables';

In [0]:
from pyspark.sql import functions as F

lake = "abfss://bronze@stflightopsdevuv0zba.dfs.core.windows.net"
source_path     = f"{lake}/bts/"
schema_location = f"{lake}/_autoloader/bts_flights/schema"
checkpoint      = f"{lake}/_autoloader/bts_flights/checkpoint"

bronze_df = (spark.readStream
    .format("cloudFiles")                            # Auto Loader
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "*.csv")               # skip readme.html files
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "false")  # bronze stays raw: all strings
    .option("cloudFiles.partitionColumns", "")       # avoid clash: folder year= vs CSV Year column
    .load(source_path)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp()))

(bronze_df.writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)                      # process everything new, then stop
    .toTable("flightops_dev.bronze.bts_flights")
    .awaitTermination())

In [0]:
%sql
SELECT Month, COUNT(*) AS flights, COUNT(DISTINCT _source_file) AS files
FROM flightops_dev.bronze.bts_flights
GROUP BY Month
ORDER BY CAST(Month AS INT);

In [0]:
   print("Total rows:", spark.table("flightops_dev.bronze.bts_flights").count())